# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a structured guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Schema URL**: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
- **Dataset Identifier**: 10.71728/senscience.y7m0-f273

This dataset contains ordered logistic regression outputs (log likelihood values across iterations, coefficients, standard errors, p-values) for variables affecting household adoption of indigenous and modern knowledge in rangeland management interventions. The data covers socio-demographic characteristics, knowledge management processes, and intervention outcomes among pastoral households in Samburu, Isiolo, and Marsabit counties, Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict or list

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate the available record sets, fields, and columns using their `@id`s. This ensures unique referencing for all dataset entities.


In [ ]:
# Retrieve record sets
record_sets = list(dataset.record_sets())  # Each is an mlcroissant.RecordSet object

print('Available Record Sets:')
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    # List fields for this record set
    fields = rs.fields
    print(f"  Fields:")
    for field in fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        # List columns if present
        columns = getattr(field, 'columns', [])
        if columns:
            print("      Columns:")
            for col in columns:
                print(f"        - Column @id: {col.id}, name: {col.name}, dataType: {col.data_type}")
    print("") # Blank line for readability

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Select one or more record sets of interest (by their `@id`) and load their records.

In [ ]:
# Extract data from all record sets
dataframes = {}

selected_record_sets = [rs.id for rs in record_sets]

for record_set_id in selected_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet @id: {record_set_id} - shape: {df.shape}")

# Display columns from the first available record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for RecordSet @id {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section demonstrates filtering, normalization, and grouping using fields referenced by their `@id`.

In [ ]:
# Choose a record set (by @id) that contains numeric fields
if dataframes:
    # For demonstration, use the first available record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    # Identify numeric fields (columns)
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Select the first numeric field by @id (column name)
        print(f"Using numeric field @id: {numeric_field}")

        threshold = df[numeric_field].mean()  # Use mean as threshold for demo
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a categorical field
        categorical_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        group_field = categorical_fields[0] if categorical_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_field)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields detected in the record set. EDA cannot proceed.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration: plot the distribution of a numeric field referenced by its `@id` and explore relationships with a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization using the filtered DataFrame
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if numeric_fields:
        numeric_field = numeric_fields[0]

        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field} (@id)')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        # If a categorical field exists, show boxplot
        categorical_fields = [col for col in df.columns if df[col].dtype == 'object']
        if categorical_fields:
            group_field = categorical_fields[0]
            plt.figure(figsize=(8, 4))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f'{numeric_field} by {group_field} (@id)')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
else:
    print("Visualization skipped: data not available.")

## 6. Conclusion
- This notebook demonstrated loading and exploring the FAIR² rangeland management dataset using `mlcroissant`, referencing all entities by their `@id`.
- We overviewed record sets, fields, and columns, extracted data, performed basic EDA, and visualized sample distributions.
- The dataset provides valuable insights into predictors for the adoption of indigenous and modern rangeland management knowledge in Northern Kenya. Missing values and selection bias should be considered in analysis.

Further directions:
- Extend EDA to correlation analysis and regression modeling using extracted features.
- Integrate domain knowledge for richer interpretability.